In [ ]:
import os
import math
import pandas as pd
import numpy as np
from pathlib import Path
import xml.dom.minidom 
from datetime import datetime, timedelta
import xml.etree.ElementTree as Et

# paga o caminho para a raiz database
def get_xml_root():
    try:
        BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        BASE_DIR = os.getcwd()
    return os.path.join(BASE_DIR, "..", "OhioT1DM")

# paga o caminho para os arquivos XML
def get_XMLs(root):
    p = Path(root)
    lista_XML = []

    for x in p.iterdir():
        if x.is_dir() :
            lista_XML.extend(get_XMLs(x))
        # elif "testing" in x.name: 
        #     continue
        # elif "training" in x.name: 
        #     continue
        elif x.suffix == ".xml": 
            lista_XML.append(x)

    return lista_XML

# estrai as informaçõens dos XMLs
def get_info(file_XML, dados: dict):
    domtree = xml.dom.minidom.parse(str(file_XML))

    patient = domtree.documentElement
    assert patient is not None

    id_patient = int(patient.getAttribute('id'))
    
    if id_patient not in dados.keys():
        dados[id_patient] = {}

    finger_stick = patient.getElementsByTagName('finger_stick')[0].getElementsByTagName('event')
    for event in finger_stick:
        ts =  binning(event.getAttribute('ts'))
        dados = new_entry(ts, dados, id_patient)
        dados[id_patient][ts]["metodo_medida"] = "finger_stick"
        dados[id_patient][ts]["glucose_level"] = int(event.getAttribute('value'))

    glucose_level = patient.getElementsByTagName('glucose_level')[0].getElementsByTagName('event')
    for event in glucose_level:
        ts =  binning(event.getAttribute('ts'))
        dados = new_entry(ts, dados, id_patient)
        dados[id_patient][ts]["metodo_medida"] = "CGM"
        dados[id_patient][ts]["glucose_level"] = int(event.getAttribute('value'))           

    bolus = patient.getElementsByTagName('bolus')[0].getElementsByTagName('event')
    for event in bolus:
        ts =  binning(event.getAttribute('ts_begin'))
        dados = new_entry(ts, dados, id_patient)
        dados[id_patient][ts]["bolus"] = event.getAttribute('dose')
        dados[id_patient][ts]["bolus_bwz_carb_input"] = event.getAttribute('bwz_carb_input')
        
    meal = patient.getElementsByTagName('meal')[0].getElementsByTagName('event')
    for event in meal:
        ts =  binning(event.getAttribute('ts'))
        dados = new_entry(ts, dados, id_patient)
        dados[id_patient][ts]["meal_type"] = event.getAttribute('type')
        dados[id_patient][ts]["meal_carbs"] = event.getAttribute('carbs')

    basal = patient.getElementsByTagName('basal')[0].getElementsByTagName('event')
    for event in basal:
        ts =  binning(event.getAttribute('ts'))

        for entry in dados[id_patient]:
            if entry >= ts:
                dados[id_patient][entry]["basal"] = float(event.getAttribute('value'))

    temp_basal = patient.getElementsByTagName('temp_basal')[0].getElementsByTagName('event')
    for event in temp_basal:
        ts_begin =  binning(event.getAttribute('ts_begin'))
        ts_end =  binning(event.getAttribute('ts_end'))

        for entry in dados[id_patient]:
            if entry >= ts_begin and entry <= ts_end:
                dados[id_patient][entry]["basal"] = float(event.getAttribute('value'))

    sleep = patient.getElementsByTagName('sleep')[0].getElementsByTagName('event')
    for event in sleep:
        ts_begin =  binning(event.getAttribute('ts_begin'))
        ts_end =  binning(event.getAttribute('ts_end'))

        for entry in dados[id_patient]:
            if entry >= ts_begin and entry <= ts_end:
                dados[id_patient][entry]["sleeping"] = True
                dados[id_patient][entry]["sleep_quality"] = event.getAttribute('quality')

    exercise = patient.getElementsByTagName('exercise')[0].getElementsByTagName('event')
    for event in exercise:
        ts_begin =  binning(event.getAttribute('ts'))
        ts_end = ts_begin + timedelta(minutes= int(event.getAttribute('duration')))

        for entry in dados[id_patient]:
            if entry >= ts and entry <= ts_end:
                dados[id_patient][entry]["doing_exercise"] = True
                dados[id_patient][entry]["exercise_intensity"] = event.getAttribute('intensity')

    
    return dados

# cria uma nova entrada caso ela não existir
def new_entry(ts, dados: dict, id_patient):
    if ts not in dados[id_patient].keys():
        dados[id_patient][ts] = {"metodo_medida": None, 
                                 "glucose_level": None,
                                 "basal": None,
                                 "bolus": None,
                                 "bolus_bwz_carb_input": None,
                                 "meal_type": None, 
                                 "meal_carbs": None,
                                 "sleeping": False,
                                 "sleep_quality": None,
                                 "exercise_intensity": None,
                                 "doing_exercise": False,
                                 "id_patient": id_patient}
    return dados

# aredonda o horario para o 5 muinutos enterior 
def binning(ts): 
    data = datetime.strptime(ts, "%d-%m-%Y %H:%M:%S")
    i = 5
    mim = data.minute//i*i
    return data.replace(minute = mim, second = 0)
    
lista_XML = get_XMLs(get_xml_root())

para = 0
dados = {}
for file in lista_XML:
    # if para > 0:
    #     break 
    dados = get_info(file, dados)
    # para +=1

for pacient in dados:
    df_paciente = pd.DataFrame(dados[pacient])
    display(df_paciente)

559


,2022-01-18 06:25:00,2022-01-18 16:55:00,2022-01-18 17:55:00,2022-01-19 04:30:00,2022-01-19 05:30:00,2022-01-19 06:25:00,2022-01-19 18:35:00,2022-01-20 04:25:00,2022-01-20 05:25:00,2022-01-20 06:25:00,...,2021-12-16 05:45:00,2021-12-20 17:20:00,2021-12-20 21:45:00,2021-12-21 05:50:00,2021-12-26 07:05:00,2022-01-07 06:15:00,2022-01-08 06:30:00,2022-01-08 12:15:00,2022-01-11 05:45:00,2022-01-12 18:00:00
metodo_medida,CGM,CGM,finger_stick,CGM,CGM,CGM,finger_stick,CGM,CGM,CGM,...,None,None,None,None,None,None,None,None,None,None
glucose_level,249,150,134,96,161,180,167,265,273,216,...,None,None,None,None,None,None,None,None,None,None
basal,0.83,0.83,0.83,0.83,0.83,0.83,0.83,0.83,0.83,0.83,...,0.73,0.88,1.25,0.73,1.25,1.15,1.15,0.9,1.15,0.83
bolus,6.3,None,0.2,None,1.2,None,3.3,2.9,None,None,...,None,None,None,None,None,None,None,None,None,None
bolus_bwz_carb_input,40,None,0,None,0,None,30,30,None,None,...,None,None,None,None,None,None,None,None,None,None
meal_type,None,None,None,None,None,None,None,None,None,None,...,Breakfast,Dinner,Snack,Breakfast,Breakfast,Breakfast,Breakfast,Snack,Breakfast,Dinner
meal_carbs,None,None,None,None,None,None,None,None,None,None,...,50,60,24,45,30,20,20,16,30,65
sleeping,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
sleep_quality,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
exercise_intensity,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


563


,2021-10-29 03:50:00,2021-10-29 05:45:00,2021-10-29 07:00:00,2021-10-29 11:20:00,2021-10-29 11:40:00,2021-10-29 12:10:00,2021-10-29 14:10:00,2021-10-29 17:10:00,2021-10-29 20:15:00,2021-10-29 20:45:00,...,2021-10-28 23:40:00,2021-10-28 23:45:00,2021-10-28 23:50:00,2021-10-28 23:55:00,2021-09-19 06:00:00,2021-09-25 12:00:00,2021-10-02 07:00:00,2021-10-05 11:30:00,2021-10-08 19:30:00,2021-10-21 20:20:00
metodo_medida,CGM,CGM,CGM,CGM,CGM,CGM,CGM,CGM,CGM,CGM,...,CGM,CGM,CGM,CGM,None,None,None,None,None,None
glucose_level,209,148,111,166,160,190,152,195,133,142,...,254,250,246,240,None,None,None,None,None,None
basal,0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8,...,0.8,0.8,0.8,0.8,0.7,0.7,1.55,0.8,0.8,0.8
bolus,6.5,3.6,11.4,None,14.8,12.4,8.8,7.5,None,13.8,...,None,None,None,None,None,None,None,None,None,None
bolus_bwz_carb_input,0,18,57,None,52,62,44,0,None,69,...,None,None,None,None,None,None,None,None,None,None
meal_type,None,None,None,None,None,None,None,None,None,None,...,Lunch,None,None,None,Breakfast,Lunch,Breakfast,Lunch,Dinner,Dinner
meal_carbs,None,None,None,None,None,None,None,None,None,None,...,45,None,None,None,14,25,25,35,15,20
sleeping,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
sleep_quality,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
exercise_intensity,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


570


...

,2027-02-13 08:55:00,2027-02-13 10:40:00,2027-02-13 12:30:00,2027-02-13 12:40:00,2027-02-13 18:30:00,2027-02-13 20:45:00,2027-02-13 23:50:00,2027-02-14 00:00:00,2027-02-14 09:05:00,2027-02-14 12:00:00,...,2027-02-03 09:20:00,2027-02-03 13:40:00,2027-02-03 19:20:00,2027-02-05 13:25:00,2027-02-06 12:15:00,2027-02-07 15:45:00,2026-12-31 10:40:00,2027-01-07 11:05:00,2027-01-14 11:30:00,2027-01-18 12:25:00
metodo_medida,CGM,CGM,finger_stick,CGM,CGM,CGM,CGM,CGM,CGM,finger_stick,...,None,None,None,None,None,None,None,None,None,None
glucose_level,116,241,200,200,181,194,149,148,153,100,...,None,None,None,None,None,None,None,None,None,None
basal,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.15,1.0,1.15
bolus,14.1,None,17.4,None,20.8,9.0,4.0,None,16.9,0.8,...,15.0,15.5,10.5,9.0,10.0,8.5,None,None,None,None
bolus_bwz_carb_input,,None,,None,,,,None,,,...,,,,,,,None,None,None,None
meal_type,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,Snack,Lunch,Lunch,Lunch
meal_carbs,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,60,95,87,75
sleeping,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
sleep_quality,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
exercise_intensity,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


584


,2025-06-29 04:05:00,2025-06-29 16:45:00,2025-06-30 05:20:00,2025-06-30 16:20:00,2025-07-01 04:35:00,2025-07-01 17:05:00,2025-07-02 06:00:00,2025-07-02 17:05:00,2025-07-03 05:25:00,2025-07-03 16:25:00,...,2025-05-20 18:55:00,2025-05-22 06:30:00,2025-05-27 18:00:00,2025-06-04 06:00:00,2025-06-09 18:15:00,2025-06-10 07:40:00,2025-06-17 06:30:00,2025-06-25 19:55:00,2025-06-26 20:30:00,2025-06-27 18:30:00
metodo_medida,finger_stick,finger_stick,finger_stick,CGM,finger_stick,finger_stick,finger_stick,CGM,finger_stick,CGM,...,None,None,None,None,None,None,None,None,None,None
glucose_level,262,170,172,138,172,223,123,172,152,109,...,None,None,None,None,None,None,None,None,None,None
basal,1.65,1.65,1.65,1.65,1.65,1.65,1.65,1.65,1.65,1.65,...,1.65,1.65,1.65,1.65,1.65,1.65,1.65,1.65,1.65,1.65
bolus,7.5,None,1.4,None,1.4,4.0,None,None,0.7,None,...,None,None,None,None,None,None,None,None,None,None
bolus_bwz_carb_input,,None,,None,,,None,None,,None,...,None,None,None,None,None,None,None,None,None,None
meal_type,None,None,None,None,None,None,None,None,None,None,...,Dinner,Breakfast,Dinner,Breakfast,Dinner,Breakfast,Breakfast,Dinner,Snack,Dinner
meal_carbs,None,None,None,None,None,None,None,None,None,None,...,60,60,60,60,60,60,60,60,15,60
sleeping,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
sleep_quality,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
exercise_intensity,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


596


,2027-05-27 05:55:00,2027-05-27 10:25:00,2027-05-27 21:25:00,2027-05-28 14:15:00,2027-05-28 17:40:00,2027-05-29 04:45:00,2027-05-29 16:15:00,2027-05-29 21:00:00,2027-05-30 06:20:00,2027-05-31 05:35:00,...,2027-05-23 15:20:00,2027-05-23 18:05:00,2027-05-23 20:10:00,2027-05-23 22:05:00,2027-05-24 06:35:00,2027-05-24 10:15:00,2027-05-24 12:10:00,2027-05-24 17:30:00,2027-05-24 20:50:00,2027-05-25 06:30:00
metodo_medida,CGM,CGM,CGM,finger_stick,CGM,CGM,CGM,CGM,CGM,CGM,...,None,None,None,None,None,None,None,None,None,None
glucose_level,179,66,226,183,54,146,207,120,161,167,...,None,None,None,None,None,None,None,None,None,None
basal,0.45,0.45,0.45,0.45,0.45,0.45,0.45,0.45,0.45,0.45,...,0.6,0.6,0.6,0.6,0.6,0.6,0.6,0.6,0.6,0.6
bolus,1.7,None,None,None,None,None,None,None,4.0,None,...,None,None,None,None,None,None,None,None,None,None
bolus_bwz_carb_input,,None,None,None,None,None,None,None,,None,...,None,None,None,None,None,None,None,None,None,None
meal_type,None,None,None,None,None,None,None,None,None,None,...,Snack,Dinner,Snack,HypoCorrection,Breakfast,Snack,Lunch,Dinner,Snack,Breakfast
meal_carbs,None,None,None,None,None,None,None,None,None,None,...,7,52,12,16,7,15,35,53,12,27
sleeping,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
sleep_quality,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
exercise_intensity,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [ ]:
import os
import math
import pandas as pd
import numpy as np
from pathlib import Path
import xml.dom.minidom 
from datetime import datetime, timedelta
import xml.etree.ElementTree as Et


domtree2018 = xml.dom.minidom.parse("C:/Users/PMI/Documents/TCC_Predicao_glicemia/OhioT1DM/2018/test/559-ws-testing.xml")
domtree2020 = xml.dom.minidom.parse("C:/Users/PMI/Documents/TCC_Predicao_glicemia/OhioT1DM/2020/train/567-ws-training.xml")

# patient2018 = domtree2018.documentElement 
# assert patient2018 is not None
patient2020 = domtree2020.documentElement
assert patient2020 is not None

# print(len(patient2018.getElementsByTagName('sleep')[0].getElementsByTagName('event')))
print(len(patient2020.getElementsByTagName("sleep")[0].getElementsByTagName('event')))


12
